### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [6]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from groq import Groq
load_dotenv()
GROQ_API_KEY = os.getenv("GROQKEY")
client = Groq(
    api_key=GROQ_API_KEY,
)

In [7]:
def groq_llm(system_prompt,user_input,llm="openai/gpt-oss-120b",temp=0.5,strm=False,reasoning="medium"):
    chat_completion = client.chat.completions.create(
    messages=[
        { 
            "role": "user",
            "content": user_input,
        }, 
        {"role": "system", "content": system_prompt},
    ],
    model=llm,
    temperature=temp,
    stream=strm,
    reasoning_effort=reasoning,
)

    return chat_completion.choices[0].message.content
groq_llm(system_prompt="ur helpful assistant",user_input="what is the true meaning of life?")

'The short answer is that there isn’t a single, universally‑agreed‑upon “true meaning of life.”\u202fWhat gives life meaning tends to depend on the lenses through which we look—philosophical, religious, scientific, cultural, and personal. Below are some of the most common ways people have tried to answer the question, along with a few thoughts on how you might explore it for yourself.\n\n---\n\n## 1.\u202fPhilosophical Perspectives  \n\n| School of Thought | Core Idea about Meaning | Notable Thinkers |\n|-------------------|------------------------|------------------|\n| **Existentialism** | Life has no inherent purpose; we create our own meaning through choices and authentic action. | Jean‑Paul Sartre, Albert Camus, Simone de Beauvoir |\n| **Absurdism** | The universe is indifferent, and the search for meaning is inherently “absurd,” but we can find freedom in embracing that absurdity. | Albert Camus |\n| **Nihilism** | There is no objective meaning, value, or purpose. Some see this a

In [8]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory): # defines a function to process all PDFs in a directory and its parameter can be called anything
    """Process all PDF files in the specified directory."""
    all_documents = [] # Initialize an empty list to store all document objects (pages returned by the loader)
    pdf_dir = Path(pdf_directory) 
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.rglob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files.")
    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:
            # Load the PDF file
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f" loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
    # Process all PDFs in the data directory

all_pdf_documents = process_all_pdfs("../data")

Found 0 PDF files.

Total documents loaded: 0


In [9]:
all_pdf_documents

[]

In [10]:
### Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...") # Print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [11]:
chunks = split_documents(all_pdf_documents)
chunks

Split 0 documents into 0 chunks.


[]

### embedding and vectorStoreDB

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [13]:
class EmbeddingManager:
    """Manages document generation using SentenceTransformer"""

    def __init__(self, model_name: str = "paraphrase-MiniLM-L3-v2"):
        """Initialize the EmbeddingManager
        Args:
            model_name: Huggingface model name for sentence embeddings
        """
        self.model_name = model_name #Object attribute value respectively
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts: List of strings to generate embeddings for
        Returns:
            Numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model is not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    ## Intialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: paraphrase-MiniLM-L3-v2
Model loaded successfully. Embedding dimension: 384


### VectorStore

In [14]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def _init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the VectorStore
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()})")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
        Args:
            documents: List of document objects with 'page_content' and 'metadata'
            embeddings: corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)  # Copy existing metadata
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document context
            documents_text.append(doc.page_content)

            # Embedding
            embedding_list.append(embedding.tolist())

        # Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embedding_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection now: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


In [15]:
chunks

[]

In [17]:
### Convert the text into embeddings
texts = [doc.page_content for doc in chunks]

## Generate embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 0 texts...


Batches: 0it [00:00, ?it/s]

Generated embeddings with shape: (0,)
Adding 0 documents to the vector store...
Error adding documents to vector store: 'VectorStore' object has no attribute 'collection'


AttributeError: 'VectorStore' object has no attribute 'collection'